# 03b — Version links and conflict labels

Notebook 03's final system (*03 final*) still fails two kinds of hard case: a question about the work-from-home policy
whose search returns only the replaced 2023 policy, and a question about the domestic meal allowance where both
values are retrieved (NIS 120 and NIS 150) but the model gives one. Two fixes are tested, both recorded at upload:

| fix | what it does |
|---|---|
| **supersede link** | a retrieved passage of a replaced document brings the best-matching passage of the document that replaced it |
| **conflict label** | the two passages that disagree are labelled `conflicts_with=<document>` (and bring each other); one prompt rule says to give both values |

A scoring fix is also applied: the search check of notebook 03 looked for English words (`Tuesday`, `replace`) in
Hebrew passages; their Hebrew forms are added.

**Design.**
1. **Pre-registered test** — *03 + both fixes* (both fixes) against *03 final* on **24 new questions** (8 version, 8 conflict,
   8 controls). Rule fixed before the first run: adopt only if it answers at least as many new version **and** new
   conflict questions correctly, and loses at most one control (the 8 new controls + 38 questions of notebook 02).
2. **Ablation (added after the first run, exploratory)** — the first run improved version questions and hurt
   conflict questions, so each fix is also run alone: *03 + supersede link* and *03 + conflict label*.
3. **Stability** — *03 final* is answered 3 times with the same input, to measure how many results change between
   identical runs (temperature 0 is not fully deterministic on vLLM).

**Findings.** The **supersede link** improves version questions — the same gain in this run and in two earlier
runs. The **conflict label** has no consistent effect: in this run it answered 2 more new conflict questions, in two
earlier runs it answered 2 fewer of notebook 03's conflict questions — and *03 final* alone moves by 2 on the new
conflict questions between identical runs. Recommendation: add the supersede link to the system; do not add the label.

**Before running:** a GPU runtime (≥ 24 GB). No API key is needed. **Runtime:** ≈ 15 min (installing vLLM and the model).


In [1]:
# ==== Setup ==================================================================
# Install only what a fresh Colab runtime is missing (most packages are preinstalled).
import importlib.util
import subprocess
import sys

REQUIRED = {  # pip name -> import name
    "sentence-transformers>=3.0": "sentence_transformers",
    "torch": "torch",
    "openai>=1.50": "openai",
    "seaborn": "seaborn",
    "pymupdf": "pymupdf",          # reads PDF text with its colour and font size (ingestion guard)
    "python-docx": "docx",         # the same for Word files
}
missing = [pip for pip, mod in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if missing:
    print("Installing:", ", ".join(missing), flush=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

# vLLM serves the model on this GPU, in its own virtual environment (as in notebook 02).
from pathlib import Path

VLLM_VERSION, VLLM_PYTHON = "0.30.0", "3.12"
VLLM_ENV = Path("/content" if Path("/content").is_dir() else ".") / f"vllm-{VLLM_VERSION}-env"
if not (VLLM_ENV / "bin" / "vllm").exists():
    print(f"Installing vLLM {VLLM_VERSION} with Python {VLLM_PYTHON} (a few minutes)...", flush=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    for step in (["venv", "--python", VLLM_PYTHON, str(VLLM_ENV)],
                 ["pip", "install", "--python", str(VLLM_ENV / "bin" / "python"), f"vllm=={VLLM_VERSION}"]):
        r = subprocess.run([sys.executable, "-m", "uv", *step], capture_output=True, text=True)
        if r.returncode:
            raise RuntimeError(f"uv {step[0]} failed:\n" + "\n".join((r.stdout + r.stderr).splitlines()[-30:]))
if not (VLLM_ENV / "bin" / "ninja").exists():   # vLLM compiles some GPU kernels on first use and needs ninja
    subprocess.run([sys.executable, "-m", "uv", "pip", "install", "-q", "--python", str(VLLM_ENV / "bin" / "python"),
                    "ninja"], check=True)
r = subprocess.run([str(VLLM_ENV / "bin" / "python"), "-c", "import vllm; print(vllm.__version__)"],
                   capture_output=True, text=True)
assert r.stdout.strip() == VLLM_VERSION, f"vLLM import failed:\n{(r.stdout + r.stderr)[-2000:]}"
print(f"vLLM {r.stdout.strip()} ready in {VLLM_ENV}")

Installing: pymupdf, python-docx
Installing vLLM 0.30.0 with Python 3.12 (a few minutes)...
vLLM 0.30.0 ready in /content/vllm-0.30.0-env


In [2]:
# ==== Configuration ==========================================================
import hashlib, io, json, os, random, re, subprocess, threading, time, unicodedata, urllib.parse, urllib.request
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

REPO_RAW = "https://raw.githubusercontent.com/tal-mat/knowledge-assistant/main"
CORPUS_SHA256_PREFIX = "0946af1b8e105a46"          # notebook 00's corpus
EMBED_MODEL, MODEL = "BAAI/bge-m3", "openai/gpt-oss-20b"
TOP_K, MAX_PER_DOC = 3, 2                           # notebook 03's final search
MAX_LINKED = 2                                      # at most 2 linked passages added per question
REPEATS = 3                                         # 03 final is answered 3 times: how stable is one run?
MAX_COMPLETION_TOKENS, MAX_ATTEMPTS, WORKERS = 1024, 6, 32
VLLM_PORT, VLLM_MAX_MODEL_LEN = 8000, 8192
CACHE_PATH = Path("/content/generation_cache.jsonl") if Path("/content").is_dir() else Path("generation_cache.jsonl")

import torch
assert torch.cuda.is_available(), "no GPU: choose Runtime -> Change runtime type -> a GPU"
show_md = lambda text: display(Markdown(text))
show_md(f"GPU `{torch.cuda.get_device_name(0)}` · answer cache: `{CACHE_PATH}` "
        f"({'found — notebook 03 answers are reused' if CACHE_PATH.is_file() else 'not found'})")

GPU `NVIDIA RTX PRO 6000 Blackwell Server Edition` · answer cache: `/content/generation_cache.jsonl` (not found)

## Questions

- **New (decide):** 8 version questions (the 2023 → 2025 work-from-home policy, phrased so that the old policy
  is a likely search hit), 8 conflict questions (the domestic meal allowance: NIS 120 in the expense procedure,
  NIS 150 in the travel policy) and 8 controls with one correct value each. A correct answer contains the values in
  `must` (one alternative of each group); the old value (2 days) only if marked as no longer valid.
- **Notebook 03's 24 hard cases** (reported) — with fix 1: Hebrew forms added to `ver-en-tuesday` and `ver-en-replaces`.
- **Notebook 02's 38 questions on the same documents** (no harm) — scored as in notebook 02: answered with a correct citation.

In [3]:
# ==== Questions ==============================================================
# (probe_id, group, language, question, must, old_value)
NEW = [
    ("new-ver-en-twice", "version", "en", "Am I still limited to working from home twice a week?", [["3", "three"]], ["2", "two", "twice"]),
    ("new-ver-he-twice", "version", "he", "האם עדיין מותר לעבוד מהבית רק פעמיים בשבוע?", [["3", "שלושה"]], ["2", "פעמיים", "יומיים"]),
    ("new-ver-en-remote", "version", "en", "How many remote days per week does the home-working policy allow?", [["3", "three"]], ["2", "two"]),
    ("new-ver-he-remote", "version", "he", "כמה ימי עבודה מהבית מאפשרת מדיניות העבודה מהבית?", [["3", "שלושה"]], ["2", "יומיים"]),
    ("new-ver-en-equipment", "version", "en", "What does the company provide for a home-office setup?", [["1,500", "1500"]], None),
    ("new-ver-he-equipment", "version", "he", "מה החברה נותנת לעובד לצורך ציוד לעבודה מהבית?", [["1,500", "1500"]], None),
    ("new-ver-en-valid", "version", "en", "Is the February 2023 work-from-home policy still in force?",
     [["no longer", "not in force", "not valid", "replaced", "superseded"]], None),
    ("new-ver-he-valid", "version", "he", "האם מדיניות העבודה מהבית מ-2023 עדיין תקפה?",
     [["לא בתוקף", "אינה בתוקף", "לא תקפה", "אינה תקפה", "הוחלפה", "מחליפה", "החליפה"]], None),
    ("new-con-en-food", "conflict", "en", "How much can I spend on food per day on a work trip in Israel?", [["120"], ["150"]], None),
    ("new-con-he-food", "conflict", "he", "כמה מותר להוציא על אוכל ביום בנסיעת עבודה בישראל?", [["120"], ["150"]], None),
    ("new-con-en-haifa", "conflict", "en", "I have a one-day business trip to Haifa. What is my meal budget?", [["120"], ["150"]], None),
    ("new-con-he-eilat", "conflict", "he", "אני נוסע לאילת לעבודה, מה תקציב הארוחות היומי שלי?", [["120"], ["150"]], None),
    ("new-con-en-cap", "conflict", "en", "What is the per-day food reimbursement cap for domestic business travel?", [["120"], ["150"]], None),
    ("new-con-he-entitled", "conflict", "he", "על כמה כסף לארוחות אני זכאי כשאני בנסיעת עבודה בתוך הארץ?", [["120"], ["150"]], None),
    ("new-con-en-named-exp", "conflict", "en", "According to the expense procedure v4.2, what is the meal limit for a domestic work trip?", [["120"]], None),
    ("new-con-he-named-travel", "conflict", "he", "לפי מדיניות הנסיעות 2025, מה קצובת האוכל היומית בנסיעה בארץ?", [["150"]], None),
    ("new-ctl-en-km", "control", "en", "What is the reimbursement rate per kilometre for using my own car?", [["2.1"]], None),
    ("new-ctl-he-receipts", "control", "he", "עד מתי צריך להגיש קבלות על הוצאות?", [["30"]], None),
    ("new-ctl-en-approval", "control", "en", "Above what amount does an expense need my manager's prior approval?", [["500"]], None),
    ("new-ctl-he-hotel", "control", "he", "מה תקרת המלון ללילה בנסיעה לחו״ל?", [["200"]], None),
    ("new-ctl-en-21", "control", "en", "How long before a trip abroad must it be approved?", [["21"]], None),
    ("new-ctl-he-booking", "control", "he", "דרך מי מזמינים מלון לנסיעת עבודה?", [["travel@tovana.example"]], None),
    ("new-ctl-he-office-day", "control", "he", "באיזה יום אני חייב להגיע למשרד?", [["שלישי"]], None),
    ("new-ctl-en-abroad-days", "control", "en", "How many days a year may I work from another country?", [["20"]], None),
]
# Notebook 03's hard cases — fix 1: Hebrew forms added to ver-en-tuesday and ver-en-replaces (# fix 1).
OLD_HARD = [
    ('ver-he-wfh-days', 'version', 'he', 'כמה ימים בשבוע מותר לעבוד מהבית לפי המדיניות העדכנית?', [['3', 'שלושה', 'three']], ['2', 'two']),
    ('ver-he-tuesday', 'version', 'he', 'איזה יום בשבוע הוא יום משרד חובה לכולם לפי המדיניות העדכנית?', [['שלישי']], None),
    ('ver-he-replaces', 'version', 'he', 'האם מדיניות 2025 מחליפה את מדיניות העבודה מהבית מפברואר 2023?', [['כן', 'מחליפה', 'החליפה']], None),
    ('ver-he-abroad', 'version', 'he', 'לכמה ימי עבודה בשנה מוגבלת עבודה מחו״ל במדיניות העדכנית?', [['20']], None),
    ('ver-he-grant', 'version', 'he', 'מה סכום מענק הציוד לעבודה מהבית במדיניות ההיברידית העדכנית?', [['1,500', '1500']], None),
    ('ver-he-effective', 'version', 'he', 'מתי נכנסת לתוקף מדיניות העבודה ההיברידית 2025?', [['01.09.2025', '1.9.2025', '1/9/2025', '01/09/2025', 'September 1, 2025', '1 September 2025', 'September 2025', 'ספטמבר 2025']], None),
    ('ver-en-wfh-days', 'version', 'en', 'How many work-from-home days per week does the current hybrid policy allow?', [['3', 'שלושה', 'three']], ['2', 'two']),
    ('ver-en-tuesday', 'version', 'en', 'Which weekday is a mandatory office day under the current hybrid policy?', [['Tuesday', 'שלישי']], None),
    ('ver-en-replaces', 'version', 'en', 'Does the 2025 policy replace the February 2023 work-from-home policy?', [['yes', 'replace', 'מחליפה']], None),
    ('ver-en-abroad', 'version', 'en', 'What is the yearly cap on working from abroad in the current policy?', [['20']], None),
    ('ver-en-not-two', 'version', 'en', 'Is the old limit of two WFH days per week still the current rule?', [['3', 'שלושה', 'three']], ['2', 'two']),
    ('ver-en-effective', 'version', 'en', 'When does the 2025 hybrid-work policy take effect?', [['01.09.2025', '1.9.2025', '1/9/2025', '01/09/2025', 'September 1, 2025', '1 September 2025', 'September 2025', 'ספטמבר 2025']], None),
    ('con-he-expense-named', 'conflict', 'he', 'לפי נוהל החזרי הוצאות, מה התקרה היומית לארוחות בנסיעה בארץ?', [['120']], None),
    ('con-he-travel-named', 'conflict', 'he', 'לפי מדיניות הנסיעות, מה קצובת הארוחות ליום בנסיעה בישראל?', [['150']], None),
    ('con-he-both', 'conflict', 'he', 'מה התקרה היומית לארוחות בנסיעת עבודה בארץ?', [['120'], ['150']], None),
    ('con-he-both-2', 'conflict', 'he', 'כמה אפשר לקבל החזר על ארוחה ביום עבודה מחוץ למשרד בארץ?', [['120'], ['150']], None),
    ('con-he-both-3', 'conflict', 'he', 'מה סכום הארוחות המותר בנסיעה עסקית בתוך ישראל?', [['120'], ['150']], None),
    ('con-he-expense-500', 'conflict', 'he', 'לפי נוהל ההחזרים, מאיזה סכום נדרש אישור מנהל מראש?', [['500']], None),
    ('con-en-expense-named', 'conflict', 'en', 'According to the expense reimbursement procedure, what is the daily meal cap in Israel?', [['120']], None),
    ('con-en-travel-named', 'conflict', 'en', 'According to the travel policy, what is the domestic meal allowance per day?', [['150']], None),
    ('con-en-both', 'conflict', 'en', 'What is the daily meal allowance for domestic business travel in Israel?', [['120'], ['150']], None),
    ('con-en-both-2', 'conflict', 'en', 'How much can employees claim for meals on a work trip inside Israel?', [['120'], ['150']], None),
    ('con-en-both-3', 'conflict', 'en', 'What meal reimbursement limit applies to domestic business days?', [['120'], ['150']], None),
    ('con-en-travel-hotel', 'conflict', 'en', 'According to the travel policy, what is the standard hotel cap per night?', [['200']], None),
]
OLD_MUST = {'ver-en-tuesday': [['Tuesday']], 'ver-en-replaces': [['yes', 'replace']]}   # notebook 03's version
# Notebook 02's answerable questions on the same four documents: (probe_id, set, language, kind, question, gold).
RELATED = [
    ('v3-025-he-en-natural', 'answerable', 'he', None, 'מה תקרת הארוחות בנסיעה פנים-ארצית לפי Travel Policy?', ['d21fafd760ec-002']),
    ('v3-026-en-en-natural', 'answerable', 'en', None, 'What is the domestic meal cap in the Travel Policy?', ['d21fafd760ec-002']),
    ('v3-027-he-en-natural', 'answerable', 'he', None, 'מה ה-per diem בנסיעה לחו״ל?', ['d21fafd760ec-003']),
    ('v3-028-en-en-natural', 'answerable', 'en', None, 'What is the international per diem?', ['d21fafd760ec-003']),
    ('v3-029-he-en-natural', 'answerable', 'he', None, 'מה תקרת המלון הרגילה בחו״ל?', ['d21fafd760ec-003']),
    ('v3-030-en-en-natural', 'answerable', 'en', None, 'What is the standard hotel cap abroad?', ['d21fafd760ec-003']),
    ('v3-031-he-en-natural', 'answerable', 'he', None, 'כמה ימים מראש צריך לאשר נסיעה לחו״ל?', ['d21fafd760ec-004']),
    ('v3-032-en-en-natural', 'answerable', 'en', None, 'How many days ahead must foreign travel be approved?', ['d21fafd760ec-004']),
    ('v3-033-he-en-natural', 'answerable', 'he', None, 'דרך מי מזמינים טיסות ומלונות?', ['d21fafd760ec-005']),
    ('v3-034-en-en-natural', 'answerable', 'en', None, 'Who books flights and hotels?', ['d21fafd760ec-005']),
    ('v3-051-en-he-natural', 'answerable', 'en', None, 'Does the 2025 policy replace the February 2023 WFH policy?', ['0a3104f246cc-000']),
    ('v3-052-he-he-natural', 'answerable', 'he', None, 'האם מדיניות 2025 מחליפה את מדיניות פברואר 2023?', ['0a3104f246cc-000']),
    ('v3-053-en-he-natural', 'answerable', 'en', None, 'How many WFH days are allowed under the 2025 policy?', ['0a3104f246cc-001']),
    ('v3-054-he-he-natural', 'answerable', 'he', None, 'כמה ימי עבודה מהבית מותרים במדיניות 2025?', ['0a3104f246cc-001']),
    ('v3-055-en-he-natural', 'answerable', 'en', None, 'Which day is the mandatory office day?', ['0a3104f246cc-001']),
    ('v3-056-he-he-natural', 'answerable', 'he', None, 'איזה יום הוא יום משרד חובה?', ['0a3104f246cc-001']),
    ('v3-057-en-he-natural', 'answerable', 'en', None, 'What is the WFH equipment grant amount?', ['0a3104f246cc-002']),
    ('v3-058-he-he-natural', 'answerable', 'he', None, 'מה סכום מענק הציוד לעבודה מהבית?', ['0a3104f246cc-002']),
    ('v3-059-en-he-natural', 'answerable', 'en', None, 'How many work-from-abroad days per year?', ['0a3104f246cc-003']),
    ('v3-060-he-he-natural', 'answerable', 'he', None, 'כמה ימי עבודה מחו״ל מותרים בשנה?', ['0a3104f246cc-003']),
    ('v3-067-en-he-natural', 'answerable', 'en', None, 'What is the private-car per-km rate?', ['43fb6ce705a3-002']),
    ('v3-068-he-he-natural', 'answerable', 'he', None, 'כמה משלמים לק״מ ברכב פרטי?', ['43fb6ce705a3-002']),
    ('v3-069-en-he-natural', 'answerable', 'en', None, 'What is the meal cap in the expense procedure?', ['43fb6ce705a3-002']),
    ('v3-070-he-he-natural', 'answerable', 'he', None, 'מה תקרת הארוחות בנוהל ההחזרים?', ['43fb6ce705a3-002']),
    ('v3-071-en-he-natural', 'answerable', 'en', None, 'From what amount does an expense need prior manager approval?', ['43fb6ce705a3-003']),
    ('v3-072-he-he-natural', 'answerable', 'he', None, 'מאיזה סכום נדרש אישור מנהל להוצאה?', ['43fb6ce705a3-003']),
    ('v3-073-en-he-natural', 'answerable', 'en', None, 'Within how many days must receipts be submitted?', ['43fb6ce705a3-004']),
    ('v3-074-he-he-natural', 'answerable', 'he', None, 'תוך כמה ימים מגישים קבלות?', ['43fb6ce705a3-004']),
    ('v3-097-en-he-natural', 'answerable', 'en', None, 'How many WFH days did the 2023 scanned policy allow?', ['42cda623f030-000']),
    ('v3-098-he-he-natural', 'answerable', 'he', None, 'כמה ימי בית התיר מסמך הסריקה מ-2023?', ['42cda623f030-000']),
    ('v3-099-en-he-natural', 'answerable', 'en', None, 'Is VPN mandatory off-site under the 2025 hybrid policy?', ['0a3104f246cc-004']),
    ('v3-100-he-he-natural', 'answerable', 'he', None, 'האם חובה VPN מחוץ למשרד לפי מדיניות 2025?', ['0a3104f246cc-004']),
    ('v3-109-he-en-short', 'answerable', 'he', None, 'תקרת ארוחות Travel', ['d21fafd760ec-002']),
    ('v3-110-en-en-short', 'answerable', 'en', None, 'Travel Policy meal cap', ['d21fafd760ec-002']),
    ('v3-113-en-he-paraphrase', 'answerable', 'en', None, 'What is the weekly remote-work quota in the current policy?', ['0a3104f246cc-001']),
    ('v3-114-he-he-paraphrase', 'answerable', 'he', None, 'מה מכסת העבודה מרחוק השבועית במדיניות העדכנית?', ['0a3104f246cc-001']),
    ('v3-117-en-he-paraphrase', 'answerable', 'en', None, 'Per the finance procedure, what is the daily meal maximum?', ['43fb6ce705a3-002']),
    ('v3-118-he-he-paraphrase', 'answerable', 'he', None, 'לפי נוהל הכספים, מה המקסימום ליום על ארוחות?', ['43fb6ce705a3-002']),
]
HARD = [dict(zip(['probe_id', 'group', 'language', 'question', 'must', 'old_value'], r), source='new') for r in NEW] + \
       [dict(zip(['probe_id', 'group', 'language', 'question', 'must', 'old_value'], r), source='notebook 03') for r in OLD_HARD]
REL = [{'probe_id': p, 'group': 'notebook 02', 'language': l, 'question': q, 'gold': g, 'source': 'notebook 02'}
       for p, _, l, _, q, g in RELATED]
PROBES = HARD + REL
assert len({p['probe_id'] for p in PROBES}) == len(PROBES)
display(pd.DataFrame(PROBES).groupby(['source', 'group']).size().rename('questions').to_frame())

questions
source      group                 
new         conflict             8
            control              8
            version              8
notebook 02 notebook 02         38
notebook 03 conflict            12
            version             12

In [4]:
# ==== Corpus, ingestion guard and document links =============================
import docx, pymupdf

corpus = urllib.request.urlopen(f"{REPO_RAW}/processed_data/chunks.jsonl", timeout=60).read()
assert hashlib.sha256(corpus).hexdigest().startswith(CORPUS_SHA256_PREFIX), "not notebook 00's corpus"
CHUNKS = {c["chunk_id"]: c for c in map(json.loads, corpus.decode("utf-8").splitlines())}

# Ingestion guard of notebook 03: remove lines a reader cannot see (white or < 3 pt text, Word's hidden text).
def hidden_spans(name: str, data: bytes) -> list[str]:
    out = []
    if name.endswith(".pdf"):
        for page in pymupdf.open(stream=data, filetype="pdf"):
            for b in page.get_text("dict")["blocks"]:
                for line in b.get("lines", []):
                    out += [s["text"].strip() for s in line["spans"]
                            if s["text"].strip() and (s["color"] >= 0xF0F0F0 or s["size"] < 3)]
    elif name.endswith(".docx"):
        for p in docx.Document(io.BytesIO(data)).paragraphs:
            for r in p.runs:
                rgb = r.font.color.rgb if r.font.color is not None and r.font.color.type is not None else None
                if r.text.strip() and (r.font.hidden or (rgb is not None and int(str(rgb), 16) >= 0xF0F0F0)
                                       or (r.font.size is not None and r.font.size.pt < 3)):
                    out.append(r.text.strip())
    return out

squash = lambda s: re.sub(r"\s+", " ", s).strip()
INDEX = {cid: dict(c) for cid, c in CHUNKS.items()}
for name in sorted({c["source_file"] for c in CHUNKS.values() if c.get("strategy") != "synthetic"}):
    if not name.endswith((".pdf", ".docx")):
        continue
    data = urllib.request.urlopen(f"{REPO_RAW}/raw_data/{urllib.parse.quote(name)}", timeout=60).read()
    for frag in map(squash, hidden_spans(name, data)):
        for c in INDEX.values():
            if c["source_file"] == name and frag in squash(c["text"]):
                c["text"] = "\n".join(ln for ln in c["text"].splitlines() if frag not in squash(ln)).strip()
assert not any("ignore all previous instructions" in c["text"].lower() for c in INDEX.values())

# Document links, recorded at upload by the document owner.
SUPERSEDED_BY = {"SCAN_20230215_0931.pdf": "מדיניות עבודה היברידית 2025.docx"}       # notebook 03
CONFLICT_WITH = {"43fb6ce705a3-002": "d21fafd760ec-002",    # expense procedure: meals NIS 120 a day   (fix 3)
                 "d21fafd760ec-002": "43fb6ce705a3-002"}    # travel policy:     meals NIS 150 a day
show_md(f"{len(INDEX)} chunks · guard applied · {len(SUPERSEDED_BY)} replaced document · {len(CONFLICT_WITH) // 2} recorded conflict")

200 chunks · guard applied · 1 replaced document · 1 recorded conflict

## Setups

Same index, search (top 3, at most 2 per document), model and notebook 03 prompt; they differ only in the two fixes.

| setup | supersede link | conflict label (+ link + rule) | role |
|---|:-:|:-:|---|
| *03 final* | – | – | baseline (answered 3×) |
| *03 + both fixes* | ✓ | ✓ | pre-registered |
| *03 + supersede link* | ✓ | – | ablation |
| *03 + conflict label* | – | ✓ | ablation |

At most 2 passages are added per question.


In [5]:
# ==== Search, links and prompts ==============================================
from sentence_transformers import SentenceTransformer

IDS = list(INDEX)
ctx_text = lambda c: (f"{c['context'].strip()}\n\n{c['text']}" if (c.get("context") or "").strip() else c["text"])
unit = lambda m: m / np.clip(np.linalg.norm(m, axis=1, keepdims=True), 1e-12, None)
_emb = SentenceTransformer(EMBED_MODEL, device="cuda")
DOC_VECS = unit(np.asarray(_emb.encode([ctx_text(INDEX[i]) for i in IDS])))
Q_VECS = dict(zip([p["question"] for p in PROBES], unit(np.asarray(_emb.encode([p["question"] for p in PROBES])))))
del _emb; torch.cuda.empty_cache()


def retrieve(question: str, links=()) -> list[dict]:
    """Notebook 03's top 3 (at most 2 per document), then the linked passages of the kinds in `links`."""
    sims = DOC_VECS @ Q_VECS[question]
    hits, per_doc = [], {}
    for j in np.argsort(-sims, kind="stable"):
        c = INDEX[IDS[j]]
        if per_doc.get(c["source_file"], 0) < MAX_PER_DOC:
            hits.append(c); per_doc[c["source_file"]] = per_doc.get(c["source_file"], 0) + 1
        if len(hits) == TOP_K:
            break
    if not links:
        return hits
    have, extra = {h["chunk_id"] for h in hits}, []
    for h in hits:
        new_doc = SUPERSEDED_BY.get(h["source_file"])
        if "supersede" in links and new_doc:                  # fix 2: the best passage of the replacing document
            cand = [j for j, i in enumerate(IDS) if INDEX[i]["source_file"] == new_doc and i not in have]
            if cand:
                extra.append(IDS[max(cand, key=lambda j: sims[j])])
        if "conflict" in links and h["chunk_id"] in CONFLICT_WITH:   # the passage it conflicts with
            extra.append(CONFLICT_WITH[h["chunk_id"]])
        have |= set(extra)
    return hits + [INDEX[i] for i in dict.fromkeys(e for e in extra if e not in {h["chunk_id"] for h in hits})][:MAX_LINKED]


def user_message(question: str, hits: list[dict], label: bool = False) -> str:
    def header(h):
        s = f", status=superseded by {SUPERSEDED_BY[h['source_file']]}" if h["source_file"] in SUPERSEDED_BY else ""
        if label and h["chunk_id"] in CONFLICT_WITH:          # fix 3: the conflict label
            s += f", conflicts_with={INDEX[CONFLICT_WITH[h['chunk_id']]]['source_file']}"
        return f"[{h['chunk_id']}] (source={h['source_file']}{s})"
    return f"Question:\n{question}\n\nPassages:\n" + "\n\n".join(f"{header(h)}\n{h['text']}" for h in hits)

_JSON_REPLY = """Output format (JSON only, no markdown fences):
{
  "answer": "<short answer text>",
  "citations": ["<chunk_id>", "..."],
  "no_info": false
}"""
_ROLE = "You are an internal knowledge assistant. Answer the employee question using only the retrieved passages provided below."
BASIC = f"""{_ROLE}

Rules:
- Base every factual claim on the passages.
- Reply in the same language as the question (Hebrew or English).
- Do not invent policies, numbers, or procedures that are not in the passages.

{_JSON_REPLY}

- "citations" lists the chunk_id values of passages you used.
- Set "no_info" to true only if you cannot answer from the passages; then "answer" should say there is no information and "citations" should be []."""
SYSTEM_RULES = """
- A passage marked status=superseded belongs to an older document that has been replaced. Answer from the current document. Use a superseded passage only if the question asks about the old version, and then say that it is no longer in force.
- If passages from different current documents give different values for the same thing, do not choose one: give each value with the name of its document, and cite both passages."""
LANGUAGE_RULE = """
- Write "answer" in the language of the question — an English question gets an English answer even when the passages are in Hebrew."""
SYSTEM = BASIC.replace("- Do not invent policies, numbers, or procedures that are not in the passages.",
                       "- Do not invent policies, numbers, or procedures that are not in the passages." + SYSTEM_RULES)
SYSTEM += "\n" + LANGUAGE_RULE.strip("\n")
assert SYSTEM != BASIC

CONFLICT_RULE = """
- A passage marked conflicts_with=<document> gives a value that the named current document gives differently. Unless the question names one document, give both values, each with its document, and cite both passages."""
SYSTEM_LINKS = SYSTEM.replace(LANGUAGE_RULE.strip("\n"), CONFLICT_RULE.strip("\n") + "\n" + LANGUAGE_RULE.strip("\n"))
PROMPTS = {"system": SYSTEM, "system_links": SYSTEM_LINKS}
PROMPT_SHA = {k: hashlib.sha256(v.encode()).hexdigest()[:12] for k, v in PROMPTS.items()}
assert PROMPT_SHA["system"] == "5cfaa1249ad4", "the system prompt must be notebook 03's, word for word"
BOTH = ("supersede", "conflict")
SETUPS = {
    "03 final":  {"prompt": "system",       "links": (),             "label": False},
    "03 + both fixes":  {"prompt": "system_links", "links": BOTH,           "label": True},   # pre-registered
    "03 + supersede link": {"prompt": "system",    "links": ("supersede",), "label": False},  # ablation
    "03 + conflict label": {"prompt": "system_links", "links": ("conflict",), "label": True},   # ablation
}
show_md("**Prompt rule added with the conflict label**\n```text" + CONFLICT_RULE + "\n```")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

**Prompt rule added with the conflict label**
```text
- A passage marked conflicts_with=<document> gives a value that the named current document gives differently. Unless the question names one document, give both values, each with its document, and cite both passages.
```

## Search check (fixes 1 and 2, no model)

Does the passage set contain every value a correct answer needs? Notebook 03's hard cases with its original
`CHECKS` and with fix 1, then the new questions; for both setups.

In [6]:
# ==== Search check ===========================================================
def has(text: str, value: str) -> bool:
    """A value in a text after Unicode normalisation; a number must stand alone (120 does not match 1200)."""
    t, v = (re.sub(r"\s+", " ", unicodedata.normalize("NFKC", x)).lower() for x in (text, value))
    if re.fullmatch(r"[\d.,/]+", v):
        return re.search(rf"(?<![\d.,]){re.escape(v)}(?![\d])", t) is not None
    return v in t

values_in = lambda hits, must: all(any(has("\n".join(h["text"] for h in hits), v) for v in g) for g in must)
old, new = [p for p in HARD if p["source"] == "notebook 03"], [p for p in HARD if p["source"] == "new"]
rows = []
for setup, s in SETUPS.items():
    hit = {p["probe_id"]: retrieve(p["question"], s["links"]) for p in PROBES}
    rows.append({"setup": setup,
                 f"nb 03 hard cases, original CHECKS (of {len(old)})":
                     sum(values_in(hit[p["probe_id"]], OLD_MUST.get(p["probe_id"], p["must"])) for p in old),
                 f"nb 03 hard cases, fix 1 (of {len(old)})": sum(values_in(hit[p["probe_id"]], p["must"]) for p in old),
                 f"new questions (of {len(new)})": sum(values_in(hit[p["probe_id"]], p["must"]) for p in new),
                 "passages per question": round(float(np.mean([len(h) for h in hit.values()])), 2)})
display(pd.DataFrame(rows).set_index("setup"))

,"nb 03 hard cases, original CHECKS (of 24)","nb 03 hard cases, fix 1 (of 24)",new questions (of 24),passages per question
setup,,,,
03 final,21,23,22,3.00
03 + both fixes,22,24,23,3.27
03 + supersede link,22,24,23,3.23
03 + conflict label,21,23,22,3.03


In [7]:
# ==== Generation (notebook 03's model, server and cache) =====================
from openai import OpenAI

CLIENT = OpenAI(api_key="EMPTY", base_url=f"http://localhost:{VLLM_PORT}/v1", max_retries=0, timeout=300)
CACHE = {}
if CACHE_PATH.is_file():
    CACHE = {r["key"]: r for r in map(json.loads, filter(str.strip, CACHE_PATH.read_text(encoding="utf-8").splitlines()))}
LOCK, PROC = threading.Lock(), {"p": None}


def start_server():
    for extra in ([], ["--enforce-eager"]):
        cmd = [str(VLLM_ENV / "bin" / "vllm"), "serve", MODEL, "--port", str(VLLM_PORT), "--max-model-len",
               str(VLLM_MAX_MODEL_LEN), "--gpu-memory-utilization", "0.90", "--max-num-seqs", str(2 * WORKERS), *extra]
        env = {k: v for k, v in os.environ.items() if not k.startswith("VLLM_")}
        env["PATH"] = f"{VLLM_ENV / 'bin'}:{env.get('PATH', '')}"
        PROC["p"] = subprocess.Popen(cmd, stdout=open("vllm.log", "w"), stderr=subprocess.STDOUT, env=env)
        t0 = time.perf_counter()
        while time.perf_counter() - t0 < 3600 and PROC["p"].poll() is None:
            try:
                urllib.request.urlopen(f"http://localhost:{VLLM_PORT}/v1/models", timeout=5)
                return print(f"server ready after {time.perf_counter() - t0:.0f}s")
            except Exception:
                time.sleep(10)
        PROC["p"].kill()
    if os.path.exists("vllm.log"):
        print("--- vllm.log output ---")
        print(Path("vllm.log").read_text())
        print("-----------------------")
    raise RuntimeError("vLLM did not start — see vllm.log printed above")


def parse(content: str) -> dict:
    text = re.sub(r"<think>.*?(</think>|\Z)", "", content, flags=re.S).strip()
    m = re.search(r"\{.*\}", text, re.S)
    try:
        o = json.loads(m.group(0) if m else text)
        return {"answer": str(o.get("answer", "")), "citations": [str(c) for c in o.get("citations") or []],
                "no_info": o.get("no_info") is True or str(o.get("no_info")).lower() == "true", "parse_error": False}
    except Exception:
        return {"answer": text, "citations": [], "no_info": False, "parse_error": True}


def cache_key(p: dict, setup: str, rep: int = 0) -> tuple[str, list[dict], str]:
    """Notebook 03's key for the first answer; a repeat adds its number, so it is generated afresh."""
    s = SETUPS[setup]
    hits = retrieve(p["question"], s["links"])
    user = user_message(p["question"], hits, s["label"])
    key = hashlib.sha256(json.dumps([PROMPT_SHA[s["prompt"]], MODEL, user] + ([rep] if rep else [])).encode()).hexdigest()
    return key, hits, user


def generate(p: dict, setup: str, rep: int = 0) -> dict:
    s = SETUPS[setup]
    key, hits, user = cache_key(p, setup, rep)
    rec = CACHE.get(key)
    if rec is None:
        for attempt in range(MAX_ATTEMPTS):
            try:
                r = CLIENT.chat.completions.create(model=MODEL, temperature=0, max_completion_tokens=MAX_COMPLETION_TOKENS,
                        messages=[{"role": "system", "content": PROMPTS[s["prompt"]]}, {"role": "user", "content": user}],
                        extra_body={"reasoning_effort": "low"})
                rec = {"key": key, **parse(r.choices[0].message.content or ""), "failed": False}
                break
            except Exception:
                time.sleep(min(60.0, 2.0 ** attempt) + random.random())
        else:
            rec = {"key": key, "answer": "", "citations": [], "no_info": False, "parse_error": False, "failed": True}
        if not rec["failed"]:
            with LOCK:
                CACHE[key] = rec
                with CACHE_PATH.open("a", encoding="utf-8") as f:
                    f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    return {**p, **rec, "setup": setup, "rep": rep, "hit_ids": [h["chunk_id"] for h in hits]}


calls = [(p, s, 0) for s in SETUPS for p in PROBES] + [(p, "03 final", r) for r in range(1, REPEATS) for p in PROBES]
need = any(cache_key(p, s, r)[0] not in CACHE for p, s, r in calls)
if need:
    start_server()
try:
    with ThreadPoolExecutor(WORKERS) as ex:
        ROWS = list(ex.map(lambda a: generate(*a), calls))
finally:
    if PROC["p"] is not None:
        PROC["p"].terminate()
show_md(f"{len(ROWS)} answers · failed {sum(r['failed'] for r in ROWS)} · invalid JSON {sum(r['parse_error'] for r in ROWS)}")

server ready after 180s


516 answers · failed 0 · invalid JSON 0

## Results


In [8]:
# ==== Scoring and decision ===================================================
OUTDATED = ("old", "previous", "earlier", "outdated", "no longer", "superseded", "replaced", "2023",
            "ישן", "קודם", "קודמת", "הקודמת", "לא בתוקף", "הוחלף", "הוחלפה")


def correct(r: dict) -> bool:
    if r["failed"] or r["no_info"]:
        return False
    if r["source"] == "notebook 02":      # notebook 02's success: a correct citation within the passages
        return bool(r["citations"]) and set(r["citations"]) <= set(r["hit_ids"]) and bool(set(r["citations"]) & set(r["gold"]))
    a = r["answer"] or ""
    if not all(any(has(a, v) for v in g) for g in r["must"]):
        return False
    return not (r["old_value"] and any(has(a, v) for v in r["old_value"]) and not any(w in a.lower() for w in OUTDATED))


ALL = pd.DataFrame(ROWS).assign(correct=lambda d: d.apply(correct, axis=1))
df = ALL[ALL["rep"] == 0]
table = df.pivot_table(index=["source", "group"], columns="setup", values="correct", aggfunc="sum")[list(SETUPS)]
table.insert(0, "questions", df[df["setup"] == "03 final"].groupby(["source", "group"]).size())
show_md("**Correct answers per setup**")
display(table)

# --- Stability: 03 final answered REPEATS times -----------------------------------------------------------
base = ALL[ALL["setup"] == "03 final"].pivot(index="probe_id", columns="rep", values="correct")
grp = df[df["setup"] == "03 final"].set_index("probe_id")["group"]
src = df[df["setup"] == "03 final"].set_index("probe_id")["source"]
flips = base.nunique(axis=1) > 1
stab = pd.DataFrame({"questions": grp.groupby([src, grp]).size(),
                     "result changes between runs": flips.groupby([src, grp]).sum()})
for r in range(REPEATS):
    stab[f"correct, run {r + 1}"] = base[r].groupby([src, grp]).sum()
show_md(f"**Stability — 03 final answered {REPEATS} times with the same input** (temperature 0)")
display(stab)
NOISE = int(flips[[i for i in flips.index if src[i] == "new"]].sum())
show_md(f"{int(flips.sum())} of {len(flips)} questions change result between identical runs ({NOISE} of them among the "
        "new questions). A difference between setups that is not larger than this can be noise.")

# --- 03b's rule, applied to every setup (exploratory for A–D) -----------------------------------------------
new_ids = [i for i in grp.index if src[i] == "new"]
ctl = [i for i in grp.index if grp[i] in ("control", "notebook 02")]
A = df[df["setup"] == "03 final"].set_index("probe_id")["correct"]
rule_rows = []
for s in list(SETUPS)[1:]:
    B = df[df["setup"] == s].set_index("probe_id")["correct"]
    v = [i for i in new_ids if grp[i] == "version"]
    c = [i for i in new_ids if grp[i] == "conflict"]
    rule_rows.append({"setup": s,
                      "new version (vs final)": f"{int(B[v].sum())} vs {int(A[v].sum())}",
                      "new conflict (vs final)": f"{int(B[c].sum())} vs {int(A[c].sum())}",
                      "controls lost": int((A[ctl] & ~B[ctl]).sum()),
                      "passes 03b's rule": bool(B[v].sum() >= A[v].sum() and B[c].sum() >= A[c].sum()
                                                and B[ctl].sum() >= A[ctl].sum() - 1)})
show_md("**Decision rule applied to each setup** (only *03 + both fixes* was pre-registered; the other two are the ablation)")
display(pd.DataFrame(rule_rows).set_index("setup"))

# --- Answers that differ from 03 final ------------------------------------------------------------------------
wide = df.pivot(index="probe_id", columns="setup", values="correct")[list(SETUPS)]
diff = wide[wide.drop(columns="03 final").ne(wide["03 final"], axis=0).any(axis=1)]
if len(diff):
    show_md("**Questions whose result differs from 03 final in at least one setup** (✓ correct · ✗ wrong · "
            "~ = this question also changes between identical runs of 03 final)")
    marks = diff.replace({True: "✓", False: "✗"})
    marks.insert(0, "noisy", ["~" if flips[i] else "" for i in marks.index])
    display(marks)
    with pd.option_context("display.max_colwidth", None):
        display(df[df["probe_id"].isin(diff.index)].pivot(index="probe_id", columns="setup", values="answer")[list(SETUPS)])


**Correct answers per setup**

setup                    questions  03 final  03 + both fixes  \
source      group                                               
new         conflict             8         5                7   
            control              8         8                8   
            version              8         6                8   
notebook 02 notebook 02         38        38               38   
notebook 03 conflict            12        12               12   
            version             12        11               12   

setup                    03 + supersede link  03 + conflict label  
source      group                                                  
new         conflict                       5                    7  
            control                        8                    8  
            version                        7                    6  
notebook 02 notebook 02                   38                   38  
notebook 03 conflict                      12                   12  
            version                       12                   11

**Stability — 03 final answered 3 times with the same input** (temperature 0)

questions  result changes between runs  \
source      group                                                 
new         conflict             8                            2   
            control              8                            0   
            version              8                            0   
notebook 02 notebook 02         38                            0   
notebook 03 conflict            12                            0   
            version             12                            0   

                         correct, run 1  correct, run 2  correct, run 3  
source      group                                                        
new         conflict                  5               7               7  
            control                   8               8               8  
            version                   6               6               6  
notebook 02 notebook 02              38              38              38  
notebook 03 conflict                 12              12              12  
            version                  11              11              11

2 of 86 questions change result between identical runs (2 of them among the new questions). A difference between setups that is not larger than this can be noise.

**Decision rule applied to each setup** (only *03 + both fixes* was pre-registered; the other two are the ablation)

,new version (vs final),new conflict (vs final),controls lost,passes 03b's rule
setup,,,,
03 + both fixes,8 vs 6,7 vs 5,0,True
03 + supersede link,7 vs 6,5 vs 5,0,True
03 + conflict label,6 vs 6,7 vs 5,0,True


**Questions whose result differs from 03 final in at least one setup** (✓ correct · ✗ wrong · ~ = this question also changes between identical runs of 03 final)

setup,noisy,03 final,03 + both fixes,03 + supersede link,03 + conflict label
probe_id,,,,,
new-con-en-haifa,,✓,✗,✓,✗
new-con-he-eilat,,✗,✓,✗,✓
new-con-he-entitled,~,✗,✓,✗,✓
new-con-he-food,~,✗,✓,✗,✓
new-ver-en-equipment,,✗,✓,✓,✗
new-ver-he-twice,,✗,✓,✗,✗
ver-en-not-two,,✗,✓,✓,✗


setup,03 final,03 + both fixes,03 + supersede link,03 + conflict label
probe_id,,,,
new-con-en-haifa,"For a one‑day business trip to Haifa, the meal allowance is up to NIS 150 per day according to the current Travel Policy 2025, and up to NIS 120 per day according to the expense reimbursement procedure v4.2.",Your meal budget for a one‑day business trip to Haifa is NIS 150 per day.,"For a one‑day business trip to Haifa, the meal allowance is up to NIS 150 per day according to the current Travel Policy 2025, and up to NIS 120 per day according to the expense reimbursement procedure v4.2.",Your meal budget for a one‑day business trip to Haifa is NIS 150 per day.
new-con-he-eilat,"התקציב היומי למזון בנסיעה לעבודה בישראל הוא 150 ש""ח.","התקציב הארוחות היומי שלך הוא עד 120 ₪ ליום לפי נוהל החזרי הוצאות v4.2, או עד 150 ₪ ליום לפי Travel_Policy 2025.","התקציב היומי למזון בנסיעה לעבודה בישראל הוא 150 ש""ח.","התקציב הארוחות היומי שלך הוא עד 120 ₪ ליום לפי נוהל החזרי הוצאות v4.2, או עד 150 ₪ ליום לפי Travel_Policy 2025."
new-con-he-entitled,בנסיעת עבודה בתוך הארץ אתה זכאי עד 150 ₪ ליום (לפי מדיניות הנסיעות 2025).,"על פי מדיניות הנוכחית, אתה זכאי עד 150 ₪ ליום בארוחות בנסיעת עבודה בתוך הארץ. יש גם גרסה אחרת שמציינת 120 ₪ ליום, אך היא מתנגשת עם המדיניות האחרת.",בנסיעת עבודה בתוך הארץ אתה זכאי עד 150 ₪ ליום (לפי מדיניות הנסיעות 2025).,"על פי מדיניות הנוכחית, אתה זכאי עד 150 ₪ ליום בארוחות בנסיעת עבודה בתוך הארץ. יש גם גרסה אחרת שמציינת 120 ₪ ליום, אך היא מתנגשת עם המדיניות האחרת."
new-con-he-food,בנסיעת עבודה בישראל ניתן להוציא עד 150 ₪ ליום על אוכל (לפי מדיניות הנסיעות 2025).,ההסכמות השונות הן: 120 ₪ ליום (נוהל החזרי הוצאות v4.2) ו-150 ₪ ליום (Travel_Policy_2025).,בנסיעת עבודה בישראל ניתן להוציא עד 150 ₪ ליום על אוכל (לפי מדיניות הנסיעות 2025).,ההסכמות השונות הן: 120 ₪ ליום (נוהל החזרי הוצאות v4.2) ו-150 ₪ ליום (Travel_Policy_2025).
new-ver-en-equipment,"The company provides the necessary equipment for a home‑office setup from its 2026 equipment catalog. All employees are eligible for a standard laptop (LT‑100), a 27‑inch monitor (MN‑27), a USB‑C docking station (DK‑01), a wireless keyboard and mouse set (KB‑01), and headphones with a microphone (HS‑01). The catalog also lists optional items such as a 34‑inch monitor (MN‑34) and a developer laptop (LT‑200) for developers, and an ergonomic chair (CH‑01) subject to safety manager approval. Equipment can be ordered through TovanaDesk in the ‘ציוד’ category.","החברה מעניקה לעובד מענק חד‑פעמי של 1,500 ש״ח לרכישת ציוד לעבודה מהבית, והעובד יכול להזמין ציוד דרך קטלוג הציוד (למשל מחשב נייד, מסך, אביזרים וכו׳).","החברה מעניקה לעובד מענק חד‑פעמי של 1,500 ש״ח לרכישת ציוד לעבודה מהבית, והעובד יכול להזמין את הציוד דרך TovanaDesk (קטלוג ציוד מחשוב 2026).","The company provides a full home‑office kit that includes a laptop (LT‑100 for all employees), a monitor (MN‑27 for all employees), a USB‑C docking station (DK‑01), wireless keyboard and mouse (KB‑01), headphones with microphone (HS‑01) and, with safety approval, an ergonomic chair (CH‑01). All items are ordered through TovanaDesk in the ‘ציוד’ category."
new-ver-he-twice,"כן, עדיין מותר לעבוד מהבית עד 3 ימים בשבוע, ולכן פעמיים בשבוע הוא בהחלט מותר.","כן, לפי מדיניות העבודה היברידית 2025 העובדים רשאים לעבוד מהבית עד 3 ימים בשבוע.","כן, לפי מדיניות העבודה היברידית 2025 העובדים רשאים לעבוד מהבית עד 3 ימים בשבוע, ולכן עדיין מותר לעבוד מהבית פעמיים בשבוע.","כן, עדיין מותר לעבוד מהבית עד 3 ימים בשבוע, ולכן פעמיים בשבוע הוא בהחלט אפשרי."
ver-en-not-two,There is no information in the current documents about a limit of two WFH days per week.,"The current policy allows employees to work from home up to 3 days per week. The older policy limited it to 2 days per week, but that version has been superseded.","No, the current rule allows up to 3 work‑from‑home days per week.","No, the old limit of two WFH days per week is no longer in force."


## Conclusions and limits

- **Supersede link — adopt** (`link_superseded: true`). Every setup that has it answers more version questions
  (new: 7–8 of 8 vs 6; notebook 03: 12 of 12 vs 11), and `ver-en-not-two` — the question it was built for — is
  answered correctly. No control question is lost, and it adds few passages (see *passages per question*). The
  same result appeared in two earlier runs.
- **Conflict label — do not adopt.** Its effect changes sign between runs: here +2 on the new conflict questions,
  in two earlier runs −2 on notebook 03's conflict questions. The stability table shows why this cannot be read:
  *03 final* itself answers 5, 7 and 7 of the 8 new conflict questions in three identical runs. With no reliable
  gain, the label is not worth an extra prompt rule and a hand-recorded link; notebook 03's rule stays.
- **Noise** — conflict questions (two values in one answer) are the least stable group; version, control and
  notebook 02 questions did not change between identical runs.

**Limits.** One version pair and one conflict exist in this corpus, so the questions show whether a mechanism works,
not how often it helps on a real corpus. The ablation was designed after the first run and uses the same new
questions, so it is exploratory. Conflicts are recorded by hand here; in production someone (or a separate check)
must find them.


## Related work and next step

The conflict failure seen here (both values retrieved, one value given) is a known, unsolved problem in RAG.

**What research found.**
- **WikiContradict** (2024, ~250 real contradictions from Wikipedia): even when a model receives two passages that
  openly contradict each other, all tested models struggle to reflect the conflict in their answer — the same
  behaviour as here (NIS 120 and NIS 150 retrieved, one value given).
- **CONFLICTS** (*DRAGged into Conflicts*, 2025): a taxonomy of conflict types and the expected behaviour for each.
  Prompting the model to reason explicitly about possible conflicts first helps noticeably, but is far from a full
  fix — consistent with 03b, where a prompt rule helped partly and not reliably.

**Proposed solutions.**
1. **One agent per document, then an aggregator** (*MADAM-RAG*): each document is answered by a separate model call
   that sees only that document; an aggregator keeps every supported answer. Gains of a few points up to ~11 over
   other methods, at roughly 4× the tokens; the hardest benchmark (RAMDocs) stays low.
2. **Source-aware consolidation** (*Astute RAG*): the model marks consistent and conflicting information, then picks
   **one** reliable answer — not suited here, where both values must be given.
3. **Detecting conflicts at ingestion** (*ContraGen*, enterprise policies): conflicts between two different documents
   were the hardest case — the best method found about two thirds of them, human annotators about half. The
   recommendation is to combine two kinds of detector and keep human review for authority and policy conflicts.

**What it means for this system.** Research supports 03b's conclusion: a prompt change alone is not enough. Three
options (our assessment, not taken directly from the sources):
- **Enforce in code** — the conflict is recorded at upload, so when both conflicting passages are retrieved the code
  can check that both values appear and add the missing one. This does not depend on the model.
- **MADAM-RAG, cheaply** — only for questions where a conflict is detected, ask the model about each document
  separately and join the answers.
- **Resolve it in the organisation** — decide which document governs (e.g. the travel policy updates the expense
  procedure) and record it as a supersede link; the system then gives one correct answer, as it already does for
  version questions.

**Next step.** The system will try the second option: *MADAM-RAG, cheaply*. When the retrieved passages include a
recorded conflict, the model is asked once per conflicting document, using only that document's passages, and the
answers are joined, each with its document name and citations. All other questions are answered as now (03 final +
supersede link).

**Sources.**
[WikiContradict](https://arxiv.org/abs/2406.13805v1) ·
[DRAGged into Conflicts](https://arxiv.org/abs/2506.08500v2) ·
[MADAM-RAG](https://arxiv.org/html/2504.13079v2) ·
[AmbigDocs](https://arxiv.org/abs/2404.12447v2) ·
[Astute RAG](https://arxiv.org/html/2410.07176v2) ·
[ContraGen](https://arxiv.org/html/2510.03418v1) ·
[Knowledge Conflicts for LLMs: A Survey](https://arxiv.org/html/2403.08319)